In [4]:

import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("청년정책_최종.csv")


In [5]:
job_df = df[df['정책대분류명'].str.contains('일자리', na=False)].copy()

In [6]:
print(f"일자리 관련 정책 수: {len(job_df)}")

일자리 관련 정책 수: 1703


In [7]:
# 제목에 금전 키워드 포함 여부
money_keywords = ['지원', '보조', '금융', '대출', '수당', '장려금']
job_df['has_money_kw'] = job_df['정책명'].astype(str).apply(lambda x: any(kw in x for kw in money_keywords))
print("금전 키워드 포함 여부에 따른 평균 조회수:")
print(job_df.groupby('has_money_kw')['조회수'].mean())
# -> 추천 시 금전 키워드 강조 필요 있음

금전 키워드 포함 여부에 따른 평균 조회수:
has_money_kw
False     74.810645
True     194.293388
Name: 조회수, dtype: float64


In [8]:
# 서울/경기/지방 다 금전 키워드를 중요시할까?

In [9]:
# 1. 지역 그룹핑 (서울 / 경기 / 지방)
def region_group(text):
    if pd.isnull(text):
        return '기타'
    if '서울' in text:
        return '서울'
    elif '경기' in text:
        return '경기'
    else:
        return '지방'

job_df['지역'] = job_df['정책거주지역코드'].apply(region_group)

In [10]:
# 2. 조건 그룹핑 (신입 / 경력 / 기타)
def condition_group(text):
    if pd.isnull(text):
        return '기타'
    text = str(text)
    if '신입' in text or '청년' in text or '취업' in text:
        return '신입'
    elif '경력' in text or '재직' in text or '창업' in text:
        return '경력'
    else:
        return '기타'

job_df['구직조건'] = job_df['정책명'].apply(condition_group)

In [11]:
# 2. 금전 키워드 포함 여부
money_keywords = ['지원', '보조', '금융', '대출', '수당', '장려금']
job_df['has_money_kw'] = job_df['정책명'].astype(str).apply(
    lambda x: any(kw in x for kw in money_keywords)
)

In [12]:
# 3. 지역별 금전 키워드 포함 비율
money_kw_summary = job_df.groupby('지역')['has_money_kw'].mean().round(3)

In [13]:
# 4. 지역별 정책명 키워드 빈도 분석
from collections import Counter
from itertools import chain
import re

def tokenize_korean(text):
    return re.findall(r'[가-힣]{2,}', str(text))

region_keywords = {}

for region in ['서울', '경기', '지방']:
    region_data = job_df[job_df['지역'] == region]
    top_texts = region_data.sort_values(by='조회수', ascending=False).head(50)['정책명']
    words = list(chain.from_iterable(top_texts.apply(tokenize_korean)))
    word_counts = Counter(words)
    top_keywords = pd.Series(word_counts).sort_values(ascending=False).head(20)
    region_keywords[region] = top_keywords

In [14]:
# 5. 결과 정리
keyword_df = pd.concat(region_keywords, axis=1).fillna(0).astype(int)
keyword_df.columns.name = '지역'

In [16]:
# 6. 출력 (Jupyter, VSCode, 일반 파이썬 환경용)
print("✅ 1. 지역별 금전 키워드 포함 비율")
print(money_kw_summary.reset_index(name='금전키워드포함비율'))
print("\n✅ 2. 지역별 인기 키워드 TOP 20")
print(keyword_df)


✅ 1. 지역별 금전 키워드 포함 비율
   지역  금전키워드포함비율
0  경기      0.269
1  서울      0.343
2  지방      0.479

✅ 2. 지역별 인기 키워드 TOP 20
지역        서울  경기  지방
청년        15  17   8
지원        15   4  13
지원사업      10   3   8
응시료        9   0   3
사업         7   6   4
서울시        7   0   0
운영         6   0   2
어학         6   0   0
자격시험       5   0   0
모집         4  11   2
서울         4   0   0
미취업        3   0   0
자격증        3   0   0
면접         2   2   0
준비비        2   0   0
용산구        2   0   0
예비인턴       2   0   0
서비스        2   0   0
서울형        2   0   0
창업         2   0   0
경기도        0   7   0
경기청년       0   5   0
화성시        0   4   0
참여자        0   4   0
의왕시        0   3   0
성남시        0   3   0
프로그램       0   3   2
공고         0   3   0
면접수당       0   2   0
맞춤형        0   2   0
수원시        0   2   0
청년희망드림     0   2   0
대학생        0   2   0
해외진출       0   2   0
미래두배       0   0   2
청년통장       0   0   2
대전         0   0   2
시험         0   0   2
창업기업       0   0   2
패키지        0   0   2
직장적응       0   0   2
임대료 

In [17]:
# 지방이 금전 키워드를 신경 많이 쓴다?

In [19]:
from collections import Counter
from itertools import chain

# 1. 재직자 포함 여부 컬럼 생성
df["is_for_employed"] = df["정책취업요건코드"].astype(str).apply(lambda x: "재직자" in x)

In [20]:
# 2. 재직자 / 비재직자 그룹 나누기
df_employed = df[df["is_for_employed"] == True]
df_unemployed = df[df["is_for_employed"] == False]

In [21]:
# 3. 상위 30개 정책씩 가져오기 (조회수 기준)
top_employed = df_employed.sort_values("조회수", ascending=False).head(30)
top_unemployed = df_unemployed.sort_values("조회수", ascending=False).head(30)


In [22]:
# 4. 정책명을 띄어쓰기 기준으로 나눠서 단어 분석
def extract_keywords(texts):
    words = list(chain.from_iterable(texts.astype(str).str.split()))
    return pd.Series(Counter(words)).sort_values(ascending=False).head(20)

In [24]:
print("\n재직자 조건 정책 상위 키워드 TOP 20:")
print(extract_keywords(top_employed["정책명"]))


재직자 조건 정책 상위 키워드 TOP 20:
청년          5
지원사업        5
지원          3
중소기업        3
복지공유제       2
재직청년        2
및           2
2025년       2
청년통장        2
청년일자리       2
도약장려금       1
관외          1
청년통장(대전)    1
미래두배        1
사업          1
내일채움공제      1
플러스         1
청년재직자       1
남동형         1
충북행복결혼공제    1
dtype: int64


In [26]:
print("\n비재직자 조건 정책 상위 키워드 TOP 20:")
print(extract_keywords(top_unemployed["정책명"]))


비재직자 조건 정책 상위 키워드 TOP 20:
청년                4
지원                4
응시료               2
지원)               2
사업                2
지원사업              1
청년도전지원사업          1
국민내일배움카드          1
청년성장프로젝트(청년카페)    1
해외일경험             1
지원사업(WELL)        1
특별지원              1
청년월세              1
장려금(사업주           1
유연근무제             1
소상공인              1
창업응원금             1
청년안심주택            1
공급활성화(임차보증금       1
무이자지원)            1
dtype: int64


일자리 정책에 금전 키워드가 있으면 조회수가 높다(사람들이 관심을 가짐)<br>
지방이 조금더 금전 키워드를 신경 쓴다

재직자/not재직자는<br>
재직자 - 실제로 회사를 구하려는 정책을 많이봄<br>
비재직자 - 실제 일자리보다는 지원해주는 정책을 많이봄